# Identification of near-duplicate images and construction of the groups used by the grouped partition

The same photograph, or photographs that differ only marginally, appear in the data set under different file identifiers. This notebook finds them and writes three tables to `data_cropped/inspecao/`:

| File | Content |
|---|---|
| `dimensoes.csv` | one row per image: path, cultivar, size, aspect ratio, orientation, culm and view identifiers from the file name, MD5 hash |
| `duplicatas.csv` | pairs of near-duplicate images (columns `a`, `b`, `classe_a`, `classe_b`, `corr`) |
| `dimensoes_grupos.csv` | `dimensoes.csv` plus the column `grupo`, the group of each image (connected components of the graph formed by the pairs) |

**Criterion.** Two images are near-duplicates when (i) they have the same orientation (horizontal or vertical), (ii) their aspect ratios (longer side divided by shorter side) differ by less than 5%, and (iii) the Pearson correlation coefficient between their grayscale versions, resized to 256 x 64 pixels, is greater than 0.95. All pairs of images of the data set are compared. Images linked directly or through other images belong to the same group. The criterion finds copies and near-copies of the same photograph; it does not find photographs of the same billet taken from a different position or view.

**Last cells.** They measure how many images of the test subsets of the earlier image-level partitions had a near-duplicate in the training or validation subsets (the leakage reported in the paper). They need the folders of the earlier runs and of the earlier fixed partitions, which are not part of the deposited data; skip them if those folders are not available.

**Language note.** Comments and messages are in English; file names and column names (`classe`, `arquivo`, `grupo`, ...) are kept as originally generated.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab_Notebooks/four_varieties')   # root folder of the experiment

In [ ]:
!pip install -q ultralytics

In [ ]:
import os, re, glob, hashlib, pandas as pd
from PIL import Image
ROOT = "/content/drive/MyDrive/Colab_Notebooks/four_varieties/data_cropped"   # adjust if needed
OUT  = os.path.join(ROOT, "inspecao")
os.makedirs(OUT, exist_ok=True)
rows = []
for f in sorted(glob.glob(os.path.join(ROOT, "*", "*"))):
    if not f.lower().endswith((".jpg", ".jpeg", ".png")): continue
    with Image.open(f) as im: w, h = im.size; modo = im.mode
    nome = os.path.basename(f)
    m = re.match(r"IMG_([A-Za-z0-9]+)_(\d+)_(\d+)", nome)
    rows.append(dict(file=os.path.relpath(f, ROOT), classe=os.path.basename(os.path.dirname(f)),
        ext=os.path.splitext(f)[1].lower(), w=w, h=h, modo=modo, razao=max(w,h)/min(w,h),
        orient="horizontal" if w>=h else "vertical",
        colmo=m.group(2) if m else None, vista=m.group(3) if m else None,
        md5=hashlib.md5(open(f,"rb").read()).hexdigest()))
df = pd.DataFrame(rows); df.to_csv(os.path.join(OUT, "dimensoes.csv"), index=False)
print(len(df), "images"); print(df.groupby("classe").size())
print(df.groupby("classe")[["w","h","razao"]].describe().T.round(2))
print(df.groupby(["classe","orient"]).size())
print("distinct culms per cultivar:\n", df.groupby("classe")["colmo"].nunique())
print("images per culm (min/median/max):", df.groupby(["classe","colmo"]).size().describe()[["min","50%","max"]].tolist())
print("files with a repeated MD5 hash:", df.duplicated("md5", keep=False).sum())

## Near-duplicate pairs and groups

In [ ]:
import os, glob, numpy as np, pandas as pd
from PIL import Image
ROOT = "/content/drive/MyDrive/Colab_Notebooks/four_varieties/data_cropped"   # adjust if needed
RUNS = "/content/drive/MyDrive/Colab_Notebooks/four_varieties/runs_estatistica_teste_n225"   # folder with run_seed_* of the earlier image-level runs (used only in the leakage check below)
OUT  = f"{ROOT}/inspecao"
assert os.path.isfile(f"{OUT}/dimensoes.csv"), "wrong ROOT"
df = pd.read_csv(f"{OUT}/dimensoes.csv")
S = (256, 64)
def thumb(f):
    return np.asarray(Image.open(os.path.join(ROOT, f)).convert("L").resize(S), np.float32).ravel()
T = {f: thumb(f) for f in df.file}
rows = df[["file","classe","w","h","razao","orient"]].values
pairs = []
for i in range(len(rows)):
    for j in range(i+1, len(rows)):
        a, b = rows[i], rows[j]
        if a[5]==b[5] and abs(a[4]-b[4])/a[4] < 0.05:
            c = float(np.corrcoef(T[a[0]], T[b[0]])[0,1])
            if c > 0.95: pairs.append((a[0], b[0], a[1], b[1], round(c,4)))
P = pd.DataFrame(pairs, columns=["a","b","classe_a","classe_b","corr"])
P.to_csv(f"{OUT}/duplicatas.csv", index=False)
parent = {f:f for f in df.file}
def find(x):
    while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
    return x
for a,b in zip(P.a,P.b): parent[find(a)] = find(b)
df["grupo"] = df.file.map(find)
df.to_csv(f"{OUT}/dimensoes_grupos.csv", index=False)
print("pairs:", len(P), "| between different cultivars:", (P.classe_a!=P.classe_b).sum())
print("images:", len(df), "| groups:", df.grupo.nunique()); print(df.groupby("classe").grupo.nunique())

## Leakage of the earlier image-level partitions (optional)

For each earlier run (`run_seed_*/dataset_split`), counts the groups that fall in more than one subset and the groups with an image in the test subset and another image outside it.

In [ ]:
mapa = dict(zip(df.file.map(os.path.basename), df.grupo)); res = []
dirs = sorted(glob.glob(f"{RUNS}/run_seed_*/dataset_split")); assert dirs, "RUNS is wrong or contains no dataset_split"
for d in dirs:
    seed = d.split("run_seed_")[1].split("/")[0]; part = {}
    for p in ("train","val","test"):
        for f in glob.glob(f"{d}/{p}/*/*"):
            part.setdefault(mapa.get(os.path.basename(f)), set()).add(p)
    part.pop(None, None)
    res.append((seed, len(part), sum(len(s)>1 for s in part.values()), sum(("test" in s and len(s)>1) for s in part.values())))
print(pd.DataFrame(res, columns=["seed","groups","groups_in_more_than_one_subset","groups_with_an_image_in_test_and_outside"]))

## Leakage of the earlier fixed partitions (optional)

The folder `datasets_canas_com_fundo_preto` holds two earlier fixed partitions: the one made in April 2026 (the folder's parent directory) and the original one of February 2025 (the folder itself). For each, counts the images of the test and validation subsets that have a copy in another subset.

In [ ]:
import glob, os, pandas as pd
from collections import Counter
g = pd.read_csv(f"{ROOT}/inspecao/dimensoes_grupos.csv")
mapa = dict(zip(g.file.map(os.path.basename), g.grupo))
achou = glob.glob("/content/drive/**/datasets_canas_com_fundo_preto", recursive=True)
print("found:", achou)
assert achou, "folder not found; set its location manually"
base = os.path.dirname(achou[0])
for nome, raiz in [("partition of April 2026", base), ("original partition (February 2025)", achou[0])]:
    part, n = {}, {}
    for p in ("train","val","test"):
        arqs = [a for a in glob.glob(f"{raiz}/{p}/*/*") if a.lower().endswith((".jpg",".jpeg",".png"))]
        n[p] = len(arqs)
        for a in arqs:
            k = mapa.get(os.path.basename(a))
            if k is not None: part.setdefault(k, Counter())[p] += 1
    t = sum(c["test"] for c in part.values() if c["test"] and len(c) > 1)
    v = sum(c["val"] for c in part.values() if c["val"] and len(c) > 1)
    print(nome, "| images per subset:", n)
    print("  TEST images with a copy in training/validation:", t, "of", n["test"])
    print("  VALIDATION images with a copy in training/test:", v, "of", n["val"])